# Errata — train on Google Colab (free T4)

Upload the **Mac training kit** (`errata-mac-train.zip` from the Errata UI's
*Mac kit* link), train `yolo11n` on the bundled clean region-crop dataset, and
download the ONNX to copy back to the Frigate host.

Runtime → Change runtime type → **T4 GPU** (free) before starting.

In [ ]:
# 1. Install dependencies
!pip -q install ultralytics onnx onnxslim
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

In [ ]:
# 2. Upload the kit zip (choose errata-mac-train.zip)
from google.colab import files
import zipfile, glob
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall("/content/kit")
print("extracted:", name)

In [ ]:
# 3. Locate dataset and fix data.yaml path (the kit was built on macOS)
import glob, yaml
from pathlib import Path
data_yaml = sorted(glob.glob("/content/kit/**/dataset/data.yaml", recursive=True))[0]
ds = Path(data_yaml).parent.resolve()
meta = yaml.safe_load(open(data_yaml))
meta["path"] = str(ds); meta["train"] = "images/train"; meta["val"] = "images/val"
open(data_yaml, "w").write(yaml.safe_dump(meta, sort_keys=False))
print("dataset:", ds)
print("classes:", len(meta.get("names", {})), "| train imgs:", len(glob.glob(str(ds / "images/train/*.jpg"))),
      "| val imgs:", len(glob.glob(str(ds / "images/val/*.jpg"))))

In [ ]:
# 4. Train (device auto-detected; edit epochs/batch if you like)
from ultralytics import YOLO
EPOCHS, IMGSZ, BATCH = 100, 640, 16
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training on", device)
model = YOLO("yolo11n.pt")
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=2,
            amp=True, device=device, project="/content/runs", name="train", exist_ok=True)

In [ ]:
# 5. Export ONNX and convert the input to NHWC (required by Frigate's detector)
import onnx as onnx_pkg
from onnx import TensorProto, helper
onnx = Path(model.export(format="onnx", imgsz=IMGSZ))
m = onnx_pkg.load(str(onnx)); g = m.graph; inp = g.input[0]
shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
if len(shape) == 4 and shape[1] == 3:
    n, c, h, w = shape; name = inp.name; out = name + "_nchw"
    g.node.insert(0, helper.make_node("Transpose", inputs=[name], outputs=[out], perm=[0, 3, 1, 2], name=name + "_to_nchw"))
    for node in g.node[1:]:
        for i, ref in enumerate(node.input):
            if ref == name: node.input[i] = out
    inp.CopyFrom(helper.make_tensor_value_info(name, TensorProto.FLOAT, [n, h, w, c]))
    onnx_pkg.checker.check_model(m)
    tmp = onnx.with_suffix(".nhwc.tmp"); onnx_pkg.save(m, str(tmp)); tmp.replace(onnx)
    print("converted ONNX input to NHWC", shape, "->", [n, h, w, c])
else:
    print("input already", shape)

In [ ]:
# 6. Publish + download  ->  copy errata_*.onnx and labels.txt to the host's
#    object-model/models/published/  (or your trained-models/ drop folder)
import json, shutil, time
stamp = time.strftime("%Y%m%d-%H%M%S")
out = Path("/content/published"); out.mkdir(exist_ok=True)
dest = out / f"errata_{stamp}.onnx"
shutil.copyfile(onnx, dest)
if (ds / "labels.txt").is_file():
    shutil.copyfile(ds / "labels.txt", out / "labels.txt")
(out / "metrics.json").write_text(json.dumps(
    {"created_at": time.time(), "epochs": EPOCHS, "imgsz": IMGSZ, "device": device}, indent=2))
print("wrote", dest)
!cd /content && zip -qr errata-model.zip published
from google.colab import files
files.download("/content/errata-model.zip")